In [1]:
from calc_setup import *
set_design_code('ec2_2004')


Calc environment ready.


In [35]:
import pandas as pd

steel_strength_thickness_table = {
    'Min_t': [0, 16, 40, 63, 80, 100, 150, 200],
    'Max_t(inclusive)': [16, 40, 63, 80, 100, 150, 200, 250],
    'S235_fy': [235, 225, 215, 215, 215, 195, 185, 175],
    'S235_fu': [360, 360, 360, 360, 360, 350, 340, 340],
    'S275_fy': [275, 265, 255, 245, 235, 225, 215, 205],
    'S275_fu': [410, 410, 410, 410, 410, 400, 380, 380],
    'S355_fy': [355, 345, 335, 325, 315, 295, 285, 275],
    'S355_fu': [470, 470, 470, 470, 470, 450, 450, 450],
    'S450_fy': [450, 430, 410, 390, 380, 380, None, None],
    'S450_fu': [550, 550, 550, 550, 550, 530, None, None]
}

steel_strength_thickness_df = pd.DataFrame(steel_strength_thickness_table)
print(steel_strength_thickness_df)

   Min_t  Max_t(inclusive)  S235_fy  S235_fu  S275_fy  S275_fu  S355_fy  \
0      0                16      235      360      275      410      355   
1     16                40      225      360      265      410      345   
2     40                63      215      360      255      410      335   
3     63                80      215      360      245      410      325   
4     80               100      215      360      235      410      315   
5    100               150      195      350      225      400      295   
6    150               200      185      340      215      380      285   
7    200               250      175      340      205      380      275   

   S355_fu  S450_fy  S450_fu  
0      470    450.0    550.0  
1      470    430.0    550.0  
2      470    410.0    550.0  
3      470    390.0    550.0  
4      470    380.0    550.0  
5      450    380.0    530.0  
6      450      NaN      NaN  
7      450      NaN      NaN  


In [36]:
def get_steel_strength_from_thickness(
    df: pd.DataFrame,
    grade: str,
    tf_mm: float,
    tw_mm: float,
    *,
    return_fu: bool = False,
):
    """
    Returns fy (and optionally fu) based on thickness t = max(tf, tw) and grade.

    df must contain columns:
      - 'Min_t'
      - 'Max_t(inclusive)'
      - f'{grade}_fy' (e.g. 'S355_fy')
      - f'{grade}_fu' (e.g. 'S355_fu') if return_fu=True

    grade examples: 'S235', 'S275', 'S355', 'S450'
    """
    grade = grade.strip().upper()
    fy_col = f"{grade}_fy"
    fu_col = f"{grade}_fu"

    required = {"Min_t", "Max_t(inclusive)", fy_col}
    if return_fu:
        required.add(fu_col)

    missing = required - set(df.columns)
    if missing:
        raise KeyError(f"Missing required columns in df: {sorted(missing)}")

    t = float(max(tf_mm, tw_mm))

    # find matching row
    mask = (df["Min_t"] <= t) & (t <= df["Max_t(inclusive)"])
    hit = df.loc[mask]

    if hit.empty:
        raise ValueError(f"No thickness range found for t = {t} mm")

    # if somehow multiple ranges overlap, take the first (shouldn't happen if table is clean)
    row = hit.iloc[0]

    fy = row[fy_col]
    if pd.isna(fy):
        raise ValueError(f"{fy_col} is not defined for t = {t} mm (table has NaN)")

    if not return_fu:
        return t, float(fy)

    fu = row[fu_col]
    if pd.isna(fu):
        raise ValueError(f"{fu_col} is not defined for t = {t} mm (table has NaN)")

    return t, float(fy), float(fu)



In [37]:
def alpha_lt_rolled_IH(d: float, bf: float):
    """
    EC3 LTB imperfection factor alpha_LT for rolled I/H sections
    based on h/b where h=d and b=bf (both in mm).
    Returns: (alpha_LT, curve, h_over_b)
    """
    if bf <= 0:
        raise ValueError("bf must be > 0")

    h_over_b = d / bf

    # Rolled I/H: curve b if h/b <= 2, else curve c
    if h_over_b <= 2.0:
        curve = "b"
        alpha = 0.34
    else:
        curve = "c"
        alpha = 0.49

    return alpha, curve, h_over_b



In [38]:
def estimate_Iw_sym_I(*, Iz: float, D: float, TF: float) -> float:
    """
    Estimate warping constant Iw [mm^6] for a doubly-symmetric I/H section.
    Inputs:
      Iz: minor-axis second moment [mm^4]
      D : overall depth [mm]
      TF: flange thickness [mm]
    """
    hs = D - TF  # ≈ distance between shear centres of flanges
    return Iz * hs**2 / 4.0



In [39]:
labels = [lbl for _, lbl in db.search("UKB", limit=20, prefer=prefer)]
df = db.to_dataframe(labels=labels, keys=["D","BF","TF","TW","A","I33","I22"], prefer=prefer)
df


,library,label,type,designation,D,BF,TF,TW,A,I33,I22
0,BSShapes2006,UKB1016X305X487,STEEL_I_SECTION,W,1036.3,308.5,54.1,30.0,62000.0,1.021884e+10,267210000.0
1,BSShapes2006,UKB1016X305X437,STEEL_I_SECTION,W,1026.1,305.4,49.0,26.9,55700.0,9.103220e+09,234470000.0
2,BSShapes2006,UKB1016X305X393,STEEL_I_SECTION,W,1015.9,303.0,43.9,24.4,50000.0,8.075030e+09,204960000.0
3,BSShapes2006,UKB1016X305X349,STEEL_I_SECTION,W,1008.1,302.0,40.0,21.1,44500.0,7.231310e+09,184600000.0
4,BSShapes2006,UKB1016X305X314,STEEL_I_SECTION,W,999.9,300.0,35.9,19.1,40000.0,6.440630e+09,162320000.0
5,BSShapes2006,UKB1016XX305X272,STEEL_I_SECTION,W,990.1,300.0,31.0,16.5,34700.0,5.539740e+09,140040000.0
6,BSShapes2006,UKB1016X305X249,STEEL_I_SECTION,W,980.1,300.0,26.0,16.5,31700.0,4.811920e+09,117540000.0
7,BSShapes2006,UKB1016X305X222,STEEL_I_SECTION,W,970.3,300.0,21.1,16.0,28300.0,4.079610e+09,95460000.0
8,BSShapes2006,UKB914X419X388,STEEL_I_SECTION,W,921.0,420.5,36.6,21.4,49400.0,7.196350e+09,454380000.0
9,BSShapes2006,UKB914X419X343,STEEL_I_SECTION,W,911.8,418.5,32.0,19.4,43700.0,6.257800e+09,391560000.0


# **General calculation**
### Calculation Report  
**Author:** Vance Kang
**Date:** {{}}  
**Project No.:** 

---

## Calculations

### Simply supported beam with UDL


In [40]:
%%render params
L = 13        # Span length, $m$
w = 0.209      # UDL, $kN/m$
E = 200e6      # Young's modulus, $kN/m^2$ (200 GPa = $200e6 \; kN/m^2$)
I = 7.05 * 10**-6     # Second moment of area, $m^4$
x =  L/2        # Position along beam, $m$

# ----------------------------------------------------


<IPython.core.display.Latex object>

![My Image](img/simplebeam.png)

In [41]:
%%render
R = w * L / 2
M_Ed = w*L**2 / 8
delta_max = 5*w*L**4 / (384*E*I)

<IPython.core.display.Latex object>

### simply supported with 1 concentrated loads

In [42]:
%%render params
L = 6.6        # span length, ℓ (m)
P = 40       # each point load (kN)
P_ult = P * 1.5 *1.2 # factored load (kN)


# -------------------------------------------------------

<IPython.core.display.Latex object>

In [43]:
%%render
R = P_ult / 2
M_max = P_ult * L / 4

<IPython.core.display.Latex object>

# -------------------------------------------------------
### simply supported with 2 concentrated loads
# -------------------------------------------------------

In [44]:
%%render params
L = 8.3        # span length, ℓ (m)
P = 0.9       # each point load (kN)
a = 3.5        # distance of each load from the support (m)

E = 200e6      # Young's modulus (kN/m²)  200 GPa = 200e6 kN/m²
I = 8.5e-5     # second moment of area (m⁴)

x_left = 3.5   # a point with x < a   (m)
x_mid  = L/2   # a point with a < x < (L - a)   (m)
# -------------------------------------------------------



<IPython.core.display.Latex object>

![My Image](img/simplebeam2concentratedloads.png)

In [45]:
%%render
R = P # R = V = P
M_Ed = P * a # Maximum moment between loads 
delta_max = (P * a)/(24 * E * I) * (3*L**2 - 4*a**2)  # Maximum deflection at center


<IPython.core.display.Latex object>

### --- USER INPUTS ---

#### Material

In [46]:

%%render params
fy  = 355               # MPa = N/mm2 (use 235/275/355 etc)
gamma_M0 = 1.0
gamma_M1 = 1.0



<IPython.core.display.Latex object>

#### Restraint conditions

In [47]:
%%render
# LTB / geometry
L_lt = 5000             # unrestrained length (mm)
k   = 1.0               # lateral bending effective length factor (often 1.0)
k_w = 1.0               # warping effective length factor (often 1.0)

# Load application height above shear centre (mm)
# For a typical I-section with load applied at TOP flange (destabilising):
# z_g ~ distance from load point to shear centre (≈ centroid for doubly-symmetric I)
# use a simple approx first; refine later.
z_g = None  # set after you pull d, tf (example below)




<IPython.core.display.Latex object>

#### Loading conditions

In [48]:
%%render
# Moment diagram / load height factors (from NCCI SN003b tables/graphs)
C1 = 1.13 #uniformly distributed load
# C1 = 1.365  #middle point load
# C1 = 1      #uniform constant bending moment
# C1 = 2.65

# Actions / results from analysis
M_Ed =  87         # max design moment about y-y (kNm)



<IPython.core.display.Latex object>

In [49]:
results = db.search("533", limit=10, prefer=prefer)
results  # list of (library, label)



[('BSShapes2006', 'UKB533X312X272'),
 ('BSShapes2006', 'UKB533X312X219'),
 ('BSShapes2006', 'UKB533X312X182'),
 ('BSShapes2006', 'UKB533X312X150'),
 ('BSShapes2006', 'UKB533X210X138'),
 ('BSShapes2006', 'UKB533X210X122'),
 ('BSShapes2006', 'UKB533X210X109'),
 ('BSShapes2006', 'UKB533X210X101'),
 ('BSShapes2006', 'UKB533X210X92'),
 ('BSShapes2006', 'UKB533X210X82')]

In [50]:
hit = db.get_with_source("UKB533X210X101", prefer=prefer)

if hit is None:
    raise ValueError("Section not found")

print("Using library:", hit.library)
sec = hit.section
print("Using sec:", hit.label)

Using library: BSShapes2006
Using sec: UKB533X210X101


In [51]:
%%render params

# --- SECTION PROPERTIES (units: mm, N) ---
# geometry
d  = db.prop(sec, "D")     # overall depth (mm)
bf = db.prop(sec, "BF")    # flange width (mm)
tf = db.prop(sec, "TF")    # flange thickness (mm)
tw = db.prop(sec, "TW")    # web thickness (mm)

# second moments
Iy = db.prop(sec, "Iy")    # mm4 (major axis)
Iz = db.prop(sec, "Iz")    # mm4 (minor axis)

# section moduli (change keys to match your DB)
Wpl_y = db.prop(sec, "Zy")   # mm3
Wel_y = db.prop(sec, "Wy_el_pos")   # mm3

# torsion / warping (needed for Mcr)
It = db.prop(sec, "J")         # mm4  (St Venant torsion constant)






<IPython.core.display.Latex object>

In [52]:
t_used, fy = get_steel_strength_from_thickness(
    steel_strength_thickness_df,
    grade="S"+str(int(fy)),
    tf_mm=tf,
    tw_mm=tw,
)

print(f"t = {t_used:.1f} mm -> fy = {fy:.0f} MPa")


t = 17.4 mm -> fy = 345 MPa


In [53]:
%%render
thickess__section = t_used 
fy = fy          # "N/mm2 steel strength from thickness lookup table 

<IPython.core.display.Latex object>

In [54]:
Iw = estimate_Iw_sym_I(Iz=Iz, D=d, TF=tf) #mm6 estimated  (warping constant)
if z_g is None:
    z_g = d/2 - tf/2     # mm (load at top flange centroid approx)

In [55]:
%%render params
# --- section property warping constant and shear centre---
Iw = Iw # mm6  (warping constant)
z_g = z_g     # mm (load at top flange centroid approx)

# --- MATERIAL ---
E  = 210000          # N/mm2
nu = 0.3
G  = E/(2*(1+nu))    # N/mm2

# --- SECTION CLASSIFICATION (simple major-axis bending case) ---
eps = sqrt(235/fy)

hw = d - 2*tf                 # clear-ish web depth (mm) (refine if you include root radii)
c  = (bf - tw)/2              # flange outstand (mm)

lamb_f = c/tf
lamb_w = hw/tw


<IPython.core.display.Latex object>

In [56]:
# limits from Table 5.2 (common simplified bending case)
# outstand flange in compression: 9ε / 10ε / 14ε
# internal web in bending:       72ε / 83ε / 124ε
# (if you later want the full "psi" stress gradient logic, you can add it)

# flange_class = (
#     1 if lam_f <= 9*eps else
#     2 if lam_f <= 10*eps else
#     3 if lam_f <= 14*eps else
#     4
# )
# web_class = (
#     1 if lam_w <= 72*eps else
#     2 if lam_w <= 83*eps else
#     3 if lam_w <= 124*eps else
#     4
# )



In [57]:
%%render
sec_class = max(flange_class, web_class)



NameError: name 'flange_class' is not defined

In [58]:
# pick Wy based on class
# Wy = Wpl_y if sec_class <= 2 else Wel_y   # class 4 would need Weff_y
Wy = Wpl_y





In [59]:
%%render
My_Rd = Wy*fy/gamma_M0          # Nmm  --- CROSS-SECTION BENDING RESISTANCE ---
My_Rd_kNm = My_Rd/1e6            # kNm


<IPython.core.display.Latex object>

In [60]:
%%render
# --- ELASTIC CRITICAL MOMENT (LTB) ---
# SN003b general expression for doubly-symmetric sections:
# C2 = 0.454
# Mcr =  C1 * (pi**2 * E * Iz) / (k*L_lt)** 2 * (sqrt(( (k/k_w)**2 * (Iw/Iz) ) + ( (k*L_lt)**2 * G * It ) / (pi**2 * E * Iz) + (C2*z_g)**2 )- C2*z_g)
Mcr =  C1 * (pi**2 * E * Iz) / (k*L_lt)** 2 * (sqrt(( (k/k_w)**2 * (Iw/Iz) ) + ( (k*L_lt)**2 * G * It ) / (pi**2 * E * Iz) ))
Mcr_kNm = Mcr/1e6                   # kNm

<IPython.core.display.Latex object>

In [61]:
# LTB imperfection factor (depends on EC3 buckling curve you choose)
# usage with your variables
alpha_LT, lt_curve, h_over_b = alpha_lt_rolled_IH(d=d, bf=bf)
print(f"h/b = {h_over_b:.3f} -> curve {lt_curve} -> alpha_LT = {alpha_LT}")
      # treat as an input so you control the buckling curve


h/b = 2.556 -> curve c -> alpha_LT = 0.49


In [62]:
alpha_LT = alpha_LT         # treat as an input so you control the buckling curve

In [63]:
%%render


# --- NON-DIMENSIONAL SLENDERNESS + REDUCTION FACTOR ---
lamb_LT = sqrt((Wy*fy)/Mcr)
lamb_LT0 = 0.4  #0.4 for rolled sections, hot finished and cold formed, 0.2 for welded sections

beta = 0.75 #0.75 for rolled sections, hot finished and cold formed, 1.00 for welded sections

alpha_LT = alpha_LT         # determined from h/b and EC3 curves
phi_LT = 0.5*(1 + alpha_LT*(lamb_LT - lamb_LT0) + beta*lamb_LT**2)
chi_LT = 1/(phi_LT + sqrt(phi_LT**2 - beta*lamb_LT**2))
chi_LT = min(chi_LT, 1.0)

# --- LTB DESIGN BUCKLING RESISTANCE MOMENT ---
Mb_Rd = chi_LT * Wy * fy / gamma_M1     # Nmm
Mb_Rd_kNm = Mb_Rd/1e6                   # kNm

# --- UTILIZATION ---
util = M_Ed / Mb_Rd_kNm

<IPython.core.display.Latex object>